<a href="https://colab.research.google.com/github/UTD2026/Mixed_Dataset_Testing_STA/blob/main/Job4_crossdomain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Job 4 — cross-domain OOD probe (issue #5, approved rescope)

The mild-shift execution risnake green-lit: train the top-CE specialist on one in-repo domain,
evaluate on another, so 2B base stays in the routable band (~0.35–0.45) and it's deterministically
gradable with no D9 gate. Directions: **medicine→logiqa** and **logiqa→medicine**.

**Two requirements baked in (per risnake):**
1. **Within-domain reference arm at matched protocol** — each eval domain also gets its *own*
   specialist (no-shift), so every table reads **shift vs no-shift** side by side.
2. **Confident-wrong AUROC alongside routed accuracy** — the combiner's conf-wrong number is the
   headline: does the in-distribution **0.718** (medicine 2B) hold or die under genuine shift?

Machinery is the hardened shift-pivot stack: fixed-span CMP (CE over the base's exact answer
token-ids), 5-fold OOF combiner, choice grading (both domains are MCQA → short gen, no CoT
truncation). In-dist reference to beat: combiner conf-wrong **0.718**, full **0.673**.

## 0 · Knobs

In [1]:
REPO_URL="https://github.com/UTD2026/rishabh-tlm.git"
WORK="/content/tlm_job4"; LOCAL_REPO_ZIP="/content/rishabh-tlm-main.zip"; GITHUB_TOKEN_ENV="GITHUB_TOKEN"
MODEL="Qwen/Qwen3.5-2B"; MTAG="qwen3_5-2b"; RUN_GPU=True

DOMAINS=["medicine","logiqa"]                 # train a specialist on each; eval each on both
DATA_REL={"medicine":"data/AdaptEval/medicine_mcqa_random_5k.json",
          "logiqa":"data/AdaptEval/logiqa_random_5k.json"}
SWEEP_REL={"medicine":"cesplit_sweep_2026-07-02/sweep2/qwen3_5-2b/medicine",   # features.jsonl (out_ce/q_ce/mc_*)
           "logiqa":"cesplit_sweep_2026-07-02/sweep2/qwen3_5-2b/logiqa"}
TOPCE_FRAC=0.05        # specialist pool = top-5% out_ce (the CE-split recipe)
N_EVAL=1000           # eval-set size per domain (enough for AUROC; fast for MCQA)
GEN_MAXNEW=32         # MCQA answer-letter; length-invariant grading
IN_DIST_REF={"conf_wrong":0.718,"full":0.673}
print({"directions":[f"{a}->{b}" for a in DOMAINS for b in DOMAINS], "n_eval":N_EVAL})

{'directions': ['medicine->medicine', 'medicine->logiqa', 'logiqa->medicine', 'logiqa->logiqa'], 'n_eval': 1000}


## 1 · Env (Colab GPU, HF-only)

In [2]:
import os,sys,subprocess
from pathlib import Path
Path(WORK).mkdir(parents=True,exist_ok=True); os.chdir(WORK)
def pip(*a): print("pip",*a); subprocess.check_call([sys.executable,"-m","pip",*a])
if RUN_GPU:
    pip("install","-U","peft","accelerate","scipy","sympy","ninja","huggingface_hub")
    if "Qwen3.5" in MODEL: pip("install","-U","transformers[serving] @ git+https://github.com/huggingface/transformers.git@main")
    for p in ["torchaudio","torchvision","torchao"]:
        try: pip("uninstall","-y",p)
        except Exception as e: print("skip",p,e)
    for m in list(sys.modules):
        if m.startswith(("transformers","peft","torchaudio","torchvision","torchao")): del sys.modules[m]
import numpy as np, pandas as pd
print("ready")

pip install -U peft accelerate scipy sympy ninja huggingface_hub
pip install -U transformers[serving] @ git+https://github.com/huggingface/transformers.git@main
pip uninstall -y torchaudio
pip uninstall -y torchvision
pip uninstall -y torchao
ready


## 2 · Clone / upload repo + locate the cesplit-sweep features (both domains)

In [3]:
import os,subprocess,shutil,zipfile,glob
from pathlib import Path
repo_dir=Path(WORK)/"rishabh-tlm"
def _clone(u):
    t=os.environ.get(GITHUB_TOKEN_ENV,"").strip()
    uu=u.replace("https://",f"https://{t}@",1) if (t and u.startswith("https://github.com/")) else u
    return subprocess.run(["git","clone","--depth","1",uu,str(repo_dir)],text=True,stdout=subprocess.PIPE,stderr=subprocess.PIPE)
def _extract(zp,into):
    into=Path(into); into.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(zp) as zf: zf.extractall(into)
if not repo_dir.exists():
    if _clone(REPO_URL).returncode!=0:
        c=glob.glob(f"{WORK}/*.zip")+glob.glob("/content/*.zip")+([LOCAL_REPO_ZIP] if Path(LOCAL_REPO_ZIP).exists() else [])
        c=[z for z in c if "rishabh-tlm" in Path(z).name]
        if c:
            tmp=Path(WORK)/"_zip"; _extract(c[0],tmp)
            cand=tmp if (tmp/"cuda_ttl").exists() else next((p for p in tmp.rglob("*") if p.is_dir() and (p/"cuda_ttl").exists()),None)
            shutil.move(str(cand),str(repo_dir))
        else:
            from google.colab import files; up=files.upload()
            z=[n for n in up if n.endswith(".zip")][0]; tmp=Path(WORK)/"_zip"; _extract(Path(os.getcwd())/z,tmp)
            cand=tmp if (tmp/"cuda_ttl").exists() else next((p for p in tmp.rglob("*") if p.is_dir() and (p/"cuda_ttl").exists()),None)
            shutil.move(str(cand),str(repo_dir))
os.chdir(repo_dir); REPO_DIR=Path.cwd(); print("repo:",REPO_DIR)

def _find_feat(rel):
    for base in [REPO_DIR/rel]+[Path(p).parent for p in glob.glob(str(REPO_DIR/"**"/Path(rel).name/"features.jsonl"),recursive=True)]:
        if (base/"features.jsonl").exists(): return base
    return None
SWEEP={d:_find_feat(SWEEP_REL[d]) for d in DOMAINS}
DATA ={d:REPO_DIR/DATA_REL[d] for d in DOMAINS}
for d in DOMAINS:
    print(f"{d:9s} features:", SWEEP[d], "| data ok:", DATA[d].exists())
missing=[d for d in DOMAINS if not SWEEP[d]]
if missing:
    print(f"\n!! missing cesplit-sweep features for {missing}. Unzip cesplit_sweep_2026-07-02 into repo root.")
    print("   (upload cell below, or use the ce-sweep upload cell from the Job3 notebook)")

Saving rishabh-tlm-main (2).zip to rishabh-tlm-main (2).zip
repo: /content/tlm_job4/rishabh-tlm
medicine  features: None | data ok: True
logiqa    features: None | data ok: True

!! missing cesplit-sweep features for ['medicine', 'logiqa']. Unzip cesplit_sweep_2026-07-02 into repo root.
   (upload cell below, or use the ce-sweep upload cell from the Job3 notebook)


### 2b · (if needed) upload the cesplit-sweep zip

In [4]:
import glob, zipfile, os
found=glob.glob(f"{WORK}/**/cesplit*sweep*.zip",recursive=True)+glob.glob("/content/**/cesplit*sweep*.zip",recursive=True)+glob.glob(f"{os.getcwd()}/cesplit*sweep*.zip")
if any(v is None for v in SWEEP.values()):
    if not found:
        from google.colab import files
        print("Pick cesplit_sweep_2026-07-02.zip …"); up=files.upload()
        found=[str(Path(os.getcwd())/n) for n in up if n.lower().endswith(".zip") and "cesplit" in n.lower()]
    if found:
        with zipfile.ZipFile(found[0]) as zf: zf.extractall(REPO_DIR)
        SWEEP={d:_find_feat(SWEEP_REL[d]) for d in DOMAINS}
        for d in DOMAINS: print(f"{d}: {SWEEP[d]}")
    else: print("no cesplit zip found")
else:
    print("features already located — skip this cell")

Pick cesplit_sweep_2026-07-02.zip …


Saving cesplit_sweep_2026-07-02.zip to cesplit_sweep_2026-07-02.zip
medicine: /content/tlm_job4/rishabh-tlm/cesplit_sweep_2026-07-02/sweep2/qwen3_5-2b/medicine
logiqa: /content/tlm_job4/rishabh-tlm/cesplit_sweep_2026-07-02/sweep2/qwen3_5-2b/logiqa


## 3 · Helpers (fixed-span CMP, OOF combiner, choice grading)

In [5]:
import torch, json, transformers, time
import torch.nn.functional as F
import numpy as np, pandas as pd
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from tqdm.auto import tqdm
SCRIPTS=REPO_DIR/"cuda_ttl/ab_routing"; sys.path.insert(0,str(SCRIPTS)); import grading

def load_lm(path):
    tok=AutoTokenizer.from_pretrained(path,trust_remote_code=True)
    if tok.pad_token_id is None: tok.pad_token=tok.eos_token
    tok.padding_side="left"
    cfg=AutoConfig.from_pretrained(path,trust_remote_code=True)
    arch=(getattr(cfg,"architectures",None) or [""])[0]; cls=getattr(transformers,arch,None) or AutoModelForCausalLM
    return cls.from_pretrained(path,torch_dtype=torch.bfloat16,device_map="cuda").eval(), tok
def load_adapter(base_path, adir):
    b,t=load_lm(base_path); return PeftModel.from_pretrained(b,str(adir)).eval(), t
def wrap(tok,q):
    m=[{"role":"user","content":q}]
    try: return tok.apply_chat_template(m,tokenize=False,add_generation_prompt=True,enable_thinking=False)
    except TypeError: return tok.apply_chat_template(m,tokenize=False,add_generation_prompt=True)
def qof(r): return (r.get("question") or r.get("instruction") or "").strip()
def goldof(r):
    for k in ("answer","output","answers"):
        if r.get(k) not in (None,""): return str(r[k]).strip()
    return ""
def sh(cmd,log,label):
    print(f"    ├ {label}",flush=True); t=time.time()
    with open(log,"w") as f:
        p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
        for i,ln in enumerate(p.stdout):
            f.write(ln)
            if i%25==0: print(f"    │  …{label} {time.time()-t:0.0f}s",flush=True)
        if p.wait()!=0: print("".join(open(log).readlines()[-8:])); raise RuntimeError(f"{label} failed")
    print(f"    └ {label} {time.time()-t:0.0f}s",flush=True)

def hf_greedy(model,tok,recs,idxs,max_new=32,want_conf=False,desc="gen"):
    preds={}; conf={}; ans={}; side=tok.padding_side; tok.padding_side="left"
    for s in tqdm(range(0,len(idxs),16),desc=desc,leave=False):
        ch=idxs[s:s+16]
        enc=tok([wrap(tok,qof(recs[i])) for i in ch],return_tensors="pt",padding=True,truncation=True,max_length=2048).to(model.device)
        w=enc["input_ids"].shape[1]
        with torch.inference_mode():
            o=model.generate(**enc,max_new_tokens=max_new,do_sample=False,output_scores=want_conf,
                             return_dict_in_generate=want_conf,pad_token_id=tok.pad_token_id)
        seq=(o.sequences[:,w:] if want_conf else o[:,w:])
        if want_conf:
            P=torch.stack([torch.softmax(sc.float(),-1) for sc in o.scores],1); chosen=P.gather(-1,seq.unsqueeze(-1)).squeeze(-1)
        mk=(seq!=tok.pad_token_id); dec=tok.batch_decode(seq,skip_special_tokens=True)
        for bi,i in enumerate(ch):
            m=mk[bi]; preds[i]=dec[bi]; ans[i]=seq[bi][m].tolist()
            if want_conf: conf[i]=float(chosen[bi][m].mean()) if m.any() else 0.0
    tok.padding_side=side; return preds, conf, ans

def cmp_scores(model,tok,recs,ans,desc="cmp"):
    side=tok.padding_side; tok.padding_side="right"; pad=tok.pad_token_id or tok.eos_token_id; out={}
    items=list(ans.keys())
    for s in tqdm(range(0,len(items),16),desc=desc,leave=False):
        ch=items[s:s+16]; seqs=[]; sp=[]
        for i in ch:
            pids=tok.encode(wrap(tok,qof(recs[i])),add_special_tokens=False); aids=ans[i] or [pad]
            seqs.append(pids+aids); sp.append((len(pids),len(aids)))
        ml=max(len(x) for x in seqs)
        inp=torch.tensor([x+[pad]*(ml-len(x)) for x in seqs],device=model.device)
        att=torch.tensor([[1]*len(x)+[0]*(ml-len(x)) for x in seqs],device=model.device)
        with torch.inference_mode(): lg=model(input_ids=inp,attention_mask=att).logits.float()
        for bi,i in enumerate(ch):
            pl,al=sp[bi]; out[i]=float(F.cross_entropy(lg[bi][pl-1:pl+al-1],inp[bi][pl:pl+al])) if al>0 else 0.0
    tok.padding_side=side; return out

def auroc(y,s):
    y=np.asarray(y); s=np.asarray(s,float); m=~np.isnan(s); y,s=y[m],s[m]
    pos,neg=(y==1),(y==0)
    if pos.sum()==0 or neg.sum()==0: return float("nan")
    o=np.argsort(s); r=np.empty(len(s)); r[o]=np.arange(1,len(s)+1)
    return (r[pos].sum()-pos.sum()*(pos.sum()+1)/2)/(pos.sum()*neg.sum())
def _fit(X,y,l2=1.,it=800,lr=.3):
    Xb=np.c_[np.ones(len(X)),X]; w=np.zeros(Xb.shape[1])
    for _ in range(it):
        p=1/(1+np.exp(-np.clip(Xb@w,-30,30))); g=Xb.T@(p-y)/len(y); g[1:]+=l2*w[1:]/len(y); w-=lr*g
    return w
def _pr(w,X): return 1/(1+np.exp(-np.clip(np.c_[np.ones(len(X)),X]@w,-30,30)))
def oof_auroc(X,y,k=5,seed=0):
    n=len(y); idx=np.arange(n); np.random.RandomState(seed).shuffle(idx); folds=np.array_split(idx,k); oof=np.full(n,np.nan)
    for f in range(k):
        te=folds[f]; tr=np.concatenate([folds[j] for j in range(k) if j!=f]); mu=X[tr].mean(0); sd=X[tr].std(0)+1e-9
        w=_fit((X[tr]-mu)/sd,y[tr].astype(float)); oof[te]=_pr(w,(X[te]-mu)/sd)
    return auroc(y,oof), oof
def best_routed(oof_score, base_ok, adapted_ok):
    """route predicted-base-wrong items (high oof) to adapter; return best achievable routed acc + oracle."""
    order=np.argsort(-oof_score); b=np.asarray(base_ok); a=np.asarray(adapted_ok); accs=[]
    for k in range(len(order)+1):
        rt=np.zeros(len(order),bool); rt[order[:k]]=True; accs.append(np.mean(np.where(rt,a,b)))
    return max(accs), float(np.mean(np.maximum(b,a)))
print("helpers ready")

helpers ready


## 4 · Train one top-CE specialist per domain

Two specialists (medicine, logiqa), each from its own domain's top-5% out_ce pool — the CE-split
recipe, `--no-merge`. Each will be evaluated on **both** domains in §6.

In [18]:
# === §4 (fixed): non-collapsing specialists + mandatory collapse gate ===
POOL_MODE = "random"    # "random" (verified non-degenerate) | "midce" (mid-CE band) | "topce" (the collapsing one — for the A/B demo)

def select_pool(rows, mode, frac=0.05, seed=42):
    import random as _r
    oce={r["idx"]:r["out_ce"] for r in rows}; k=max(1,int(frac*len(rows)))
    if mode=="topce":  return sorted(oce,key=lambda i:-oce[i])[:k]
    if mode=="midce":  s=sorted(oce,key=lambda i:oce[i]); lo=len(s)//2-k//2; return s[lo:lo+k]
    return _r.Random(seed).sample(list(oce),k)   # random

def assert_not_collapsed(adir, domain, n=24):
    """the gate: an adapter that outputs <2 unique answers on n items is degenerate — fail loudly."""
    recs=json.load(open(DATA[domain]))[:n]
    am,at=load_adapter(MODEL,adir); ap,_,_=hf_greedy(am,at,recs,list(range(n)),max_new=GEN_MAXNEW); del am; torch.cuda.empty_cache()
    letters=[grading.extract(ap[i],grading.detect_answer_type(domain)).value for i in range(n)]
    uniq=len(set(str(l) for l in letters))
    from collections import Counter
    print(f"    gate[{domain}]: {uniq} unique answers on {n} items — {Counter(str(l) for l in letters).most_common(3)}")
    if uniq<2:
        raise RuntimeError(f"COLLAPSED adapter ({domain}, pool={POOL_MODE}): constant output. "
                           f"Do not evaluate — the routed numbers would be base+constant, not adaptation.")
    return uniq

FEATS={}; ADAPT={}
if RUN_GPU:
    for d in DOMAINS:
        rows=[json.loads(l) for l in open(SWEEP[d]/"features.jsonl")]
        FEATS[d]={r["idx"]:r for r in rows}
        sel=select_pool(rows, POOL_MODE)
        W=Path(WORK)/f"spec_{POOL_MODE}"/d; W.mkdir(parents=True,exist_ok=True)
        json.dump({"selected":[{"idx":int(i),"weight":1.0} for i in sel]},open(W/"sel.json","w"))
        adir=W/"adapter"; ADAPT[d]=adir
        if not (adir/"adapter_model.safetensors").exists():
            print(f"[{d}] training {POOL_MODE} specialist ({len(sel)} items)…",flush=True)
            sh([sys.executable,str(SCRIPTS/"ab_train_ttl.py"),"--model",MODEL,"--data",str(DATA[d]),
                "--selection-file",str(W/"sel.json"),"--max-samples","5000","--output-dir",str(adir),"--no-merge"],
               W/"train.log", f"train {d}")
        else: print(f"[{d}] specialist cached ✓")
        assert_not_collapsed(adir, d)        # <-- the gate: crashes here if it collapsed, before any eval
    print("\nall specialists passed the collapse gate ✓ — safe to run §5/§6")
else: print("RUN_GPU=False")

[medicine] training random specialist (250 items)…
    ├ train medicine
    │  …train medicine 4s
    └ train medicine 66s


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

gen:   0%|          | 0/2 [00:00<?, ?it/s]

    gate[medicine]: 4 unique answers on 24 items — [('B', 10), ('D', 7), ('A', 4)]
[logiqa] training random specialist (250 items)…
    ├ train logiqa
    │  …train logiqa 4s
    └ train logiqa 66s


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

gen:   0%|          | 0/2 [00:00<?, ?it/s]

    gate[logiqa]: 4 unique answers on 24 items — [('B', 11), ('C', 6), ('A', 4)]

all specialists passed the collapse gate ✓ — safe to run §5/§6


## 5 · Base arms per eval domain (shared across specialists)

Base greedy preds + within-model confidence + exact answer token-ids, once per eval domain.

In [19]:
BASE={}   # eval domain -> dict(bpred, within, ansids, base_ok)
if RUN_GPU:
    for e in DOMAINS:
        recs=json.load(open(DATA[e]))[:N_EVAL]; idxs=list(range(len(recs)))
        bm,btok=load_lm(MODEL)
        bpred,within,ans=hf_greedy(bm,btok,recs,idxs,max_new=GEN_MAXNEW,want_conf=True,desc=f"{e} base")
        del bm; torch.cuda.empty_cache()
        at=grading.detect_answer_type(e)
        base_ok={i:bool(grading.is_correct(grading.extract(bpred[i],at).value, goldof(recs[i]), at)) for i in idxs}
        BASE[e]=dict(recs=recs, bpred=bpred, within=within, ans=ans, base_ok=base_ok, at=at)
        print(f"[{e}] base_acc={np.mean(list(base_ok.values())):.3f}  n={len(idxs)}",flush=True)
else: print("RUN_GPU=False")

Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

medicine base:   0%|          | 0/63 [00:00<?, ?it/s]

[medicine] base_acc=0.491  n=1000


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

logiqa base:   0%|          | 0/63 [00:00<?, ?it/s]

[logiqa] base_acc=0.515  n=1000


## 6 · Cross + within eval — every (specialist × eval domain)

For each pair: adapted preds on the eval domain, CMP (specialist's surprise at base's answer),
grade, assemble signals, compute headroom + routed acc + combiner full/conf-wrong AUROC. The
within-domain pair (spec==eval) is the **no-shift reference**; spec!=eval is the **shift** arm.

In [20]:
RESULT=[]
if RUN_GPU:
    for spec in DOMAINS:
        for e in DOMAINS:
            b=BASE[e]; recs=b["recs"]; idxs=list(range(len(recs))); at=b["at"]
            am,atok=load_adapter(MODEL, ADAPT[spec])
            nz=sum(1 for n,p in am.named_parameters() if "lora_B" in n and p.abs().max()>0); assert nz>0,"adapter no-op"
            apred,_,_=hf_greedy(am,atok,recs,idxs,max_new=GEN_MAXNEW,desc=f"{spec}->{e} adapt")
            cmp=cmp_scores(am,atok,recs,b["ans"],desc=f"{spec}->{e} cmp")
            del am; torch.cuda.empty_cache()
            adp_ok={i:bool(grading.is_correct(grading.extract(apred[i],at).value, goldof(recs[i]), at)) for i in idxs}
            fe=FEATS[e]
            rows=[dict(idx=i, base_ok=b["base_ok"][i], adapted_ok=adp_ok[i], within=b["within"].get(i,np.nan),
                       cmp=cmp.get(i,np.nan),
                       q_ce=fe.get(i,{}).get("q_ce",np.nan), mc_top_p=fe.get(i,{}).get("mc_top_p",np.nan),
                       out_ce=fe.get(i,{}).get("out_ce",np.nan)) for i in idxs]
            D=pd.DataFrame(rows)
            base=D.base_ok.mean(); adaptall=D.adapted_ok.mean(); orc=(D.base_ok|D.adapted_ok).mean(); head=orc-base
            y=(~D.base_ok).astype(int).values
            use=[c for c in ["within","cmp","mc_top_p","q_ce"] if not D[c].isna().all()]
            comb_full=comb_cw=routed=np.nan; orc_routed=orc
            if len(use)>=2 and 20<y.sum()<len(y)-20:
                X=D[use].astype(float).copy()
                for c in use: X[c]=X[c].fillna(X[c].mean())
                comb_full,oof=oof_auroc(X.values,y)
                hi=D["within"].astype(float).values>=np.nanmedian(D["within"].astype(float).values)
                comb_cw=auroc(y[hi],oof[hi]) if hi.sum()>20 else np.nan
                routed,orc_routed=best_routed(oof, D.base_ok.values, D.adapted_ok.values)
            arm="no-shift" if spec==e else "SHIFT"
            RESULT.append(dict(eval=e, arm=arm, specialist=spec, base=round(base,3), adapt_all=round(adaptall,3),
                               oracle=round(orc,3), headroom=round(head,3),
                               routed_combiner=round(routed,3) if not np.isnan(routed) else np.nan,
                               routed_oracle=round(orc_routed,3),
                               comb_full=round(comb_full,3) if not np.isnan(comb_full) else np.nan,
                               comb_conf_wrong=round(comb_cw,3) if not np.isnan(comb_cw) else np.nan))
            print(f"[{spec}->{e} | {arm}] base={base:.3f} headroom={head:+.3f} "
                  f"routed={RESULT[-1]['routed_combiner']} conf_wrong={RESULT[-1]['comb_conf_wrong']}",flush=True)
    J4=pd.DataFrame(RESULT); display(J4)
else: print("RUN_GPU=False")

Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

medicine->medicine adapt:   0%|          | 0/63 [00:00<?, ?it/s]

medicine->medicine cmp:   0%|          | 0/63 [00:00<?, ?it/s]

[medicine->medicine | no-shift] base=0.491 headroom=+0.113 routed=0.528 conf_wrong=0.698


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

logiqa->logiqa adapt:   0%|          | 0/63 [00:00<?, ?it/s]

logiqa->logiqa cmp:   0%|          | 0/63 [00:00<?, ?it/s]

[logiqa->logiqa | no-shift] base=0.515 headroom=+0.035 routed=0.528 conf_wrong=0.651


,eval,arm,specialist,base,adapt_all,oracle,headroom,routed_combiner,routed_oracle,comb_full,comb_conf_wrong
0,medicine,no-shift,medicine,0.491,0.527,0.604,0.113,0.528,0.604,0.696,0.698
1,logiqa,SHIFT,medicine,0.515,0.524,0.592,0.077,0.536,0.592,0.713,0.644
2,medicine,SHIFT,logiqa,0.491,0.491,0.515,0.024,0.495,0.515,0.699,0.708
3,logiqa,no-shift,logiqa,0.515,0.521,0.550,0.035,0.528,0.550,0.709,0.651


## 7 · Shift-vs-no-shift table + verdict

Grouped by eval domain so each row-pair reads shift against its matched within-domain reference,
with the in-distribution 0.718 conf-wrong reference as the bar the headline signal must clear.

In [21]:
if RUN_GPU and RESULT:
    J4=pd.DataFrame(RESULT).sort_values(["eval","arm"])
    display(J4[["eval","arm","specialist","base","adapt_all","oracle","headroom","routed_combiner","comb_full","comb_conf_wrong"]])
    print(f"\nin-dist reference (medicine 2B, no shift): conf-wrong={IN_DIST_REF['conf_wrong']}  full={IN_DIST_REF['full']}\n")
    for e in DOMAINS:
        sub={r["arm"]:r for r in RESULT if r["eval"]==e}
        ns=sub.get("no-shift"); sh_=sub.get("SHIFT")
        if not(ns and sh_): continue
        print(f"eval={e}:")
        print(f"  no-shift ({ns['specialist']}->{e}):  base={ns['base']} headroom={ns['headroom']:+.3f} routed={ns['routed_combiner']} conf_wrong={ns['comb_conf_wrong']}")
        print(f"  SHIFT    ({sh_['specialist']}->{e}):  base={sh_['base']} headroom={sh_['headroom']:+.3f} routed={sh_['routed_combiner']} conf_wrong={sh_['comb_conf_wrong']}")
        cw=sh_["comb_conf_wrong"]
        if cw is not None and not (isinstance(cw,float) and np.isnan(cw)):
            v=("CONF-WRONG HOLDS under shift (>=0.70) — the router survives OOD" if cw>=0.70
               else f"conf-wrong {cw} < 0.70 under shift — signal does NOT survive to deployable")
            d=cw-IN_DIST_REF["conf_wrong"]; print(f"  -> shift conf-wrong {cw} vs in-dist 0.718: {d:+.3f}  |  {v}")
        print()
    print("HEADLINE for #5: does shift conf-wrong hold ~0.72 (router survives OOD) or drop (dies under shift)?")
    print("Also report: shift routed vs no-shift routed at matched protocol (the shift-vs-no-shift read risnake asked for).")
else: print("no results")

,eval,arm,specialist,base,adapt_all,oracle,headroom,routed_combiner,comb_full,comb_conf_wrong
1,logiqa,SHIFT,medicine,0.515,0.524,0.592,0.077,0.536,0.713,0.644
3,logiqa,no-shift,logiqa,0.515,0.521,0.550,0.035,0.528,0.709,0.651
2,medicine,SHIFT,logiqa,0.491,0.491,0.515,0.024,0.495,0.699,0.708
0,medicine,no-shift,medicine,0.491,0.527,0.604,0.113,0.528,0.696,0.698



in-dist reference (medicine 2B, no shift): conf-wrong=0.718  full=0.673

eval=medicine:
  no-shift (medicine->medicine):  base=0.491 headroom=+0.113 routed=0.528 conf_wrong=0.698
  SHIFT    (logiqa->medicine):  base=0.491 headroom=+0.024 routed=0.495 conf_wrong=0.708
  -> shift conf-wrong 0.708 vs in-dist 0.718: -0.010  |  CONF-WRONG HOLDS under shift (>=0.70) — the router survives OOD

eval=logiqa:
  no-shift (logiqa->logiqa):  base=0.515 headroom=+0.035 routed=0.528 conf_wrong=0.651
  SHIFT    (medicine->logiqa):  base=0.515 headroom=+0.077 routed=0.536 conf_wrong=0.644
  -> shift conf-wrong 0.644 vs in-dist 0.718: -0.074  |  conf-wrong 0.644 < 0.70 under shift — signal does NOT survive to deployable

HEADLINE for #5: does shift conf-wrong hold ~0.72 (router survives OOD) or drop (dies under shift)?
Also report: shift routed vs no-shift routed at matched protocol (the shift-vs-no-shift read risnake asked for).


In [22]:
# 1. did training converge or diverge? tail the train logs
for d in DOMAINS:
    print(f"=== {d} train tail ===")
    print("".join(open(Path(WORK)/"spec"/d/"train.log").readlines()[-12:]))

# 2. sanity: what does the adapter actually output? compare 3 base vs adapted preds
e="medicine"; recs=BASE[e]["recs"]
am,atok=load_adapter(MODEL, ADAPT["medicine"])
ap,_,_=hf_greedy(am,atok,recs,list(range(5)),max_new=32); del am; torch.cuda.empty_cache()
for i in range(5):
    print(f"base={BASE[e]['bpred'][i][:40]!r}  adapted={ap[i][:40]!r}  gold={goldof(recs[i])}")

=== medicine train tail ===
step 40/250 | loss 0.8224 | lr 4.95e-05 | 2.32 it/s
step 60/250 | loss 0.0666 | lr 4.71e-05 | 2.78 it/s
step 80/250 | loss 0.1160 | lr 4.30e-05 | 3.08 it/s
step 100/250 | loss 0.0942 | lr 3.75e-05 | 3.29 it/s
step 120/250 | loss 0.0129 | lr 3.10e-05 | 3.44 it/s
step 140/250 | loss 0.4507 | lr 2.41e-05 | 3.56 it/s
step 160/250 | loss 0.0535 | lr 1.73e-05 | 3.65 it/s
step 180/250 | loss 0.2357 | lr 1.10e-05 | 3.74 it/s
step 200/250 | loss 0.1336 | lr 5.85e-06 | 3.80 it/s
step 220/250 | loss 0.1682 | lr 2.16e-06 | 3.85 it/s
step 240/250 | loss 0.1150 | lr 2.43e-07 | 3.90 it/s
adapter -> /content/tlm_job4/spec/medicine/adapter

=== logiqa train tail ===
step 40/250 | loss 1.1242 | lr 4.95e-05 | 4.12 it/s
step 60/250 | loss 0.5867 | lr 4.71e-05 | 4.18 it/s
step 80/250 | loss 0.5020 | lr 4.30e-05 | 4.21 it/s
step 100/250 | loss 0.4754 | lr 3.75e-05 | 4.21 it/s
step 120/250 | loss 0.4405 | lr 3.10e-05 | 4.23 it/s
step 140/250 | loss 0.4876 | lr 2.41e-05 | 4.25 it/s

Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

gen:   0%|          | 0/1 [00:00<?, ?it/s]

base='C\n'  adapted='C\n'  gold=C
base='C\n'  adapted='D\n'  gold=D
base='D\n'  adapted='D\n'  gold=A
base='B\n'  adapted='B\n'  gold=B
base='C\n'  adapted='A\n'  gold=D


In [24]:
# what did the sweep's working adapters use? check its logs/configs if shipped, or the recipe doc
import glob
print(glob.glob(f"{SWEEP['medicine'].parent}/**/train*.log", recursive=True)[:3])
print(glob.glob(f"{SWEEP['medicine'].parent}/**/adapter*/adapter_config.json", recursive=True)[:3])
# and grep the recipe doc
!grep -rE "learning-rate|epochs|lora-rank|1e-5|5e-5" {REPO_DIR}/cuda_ttl/ab_routing/README.md {REPO_DIR}/docs/*.md 2>/dev/null | head

[]
['/content/tlm_job4/rishabh-tlm/cesplit_sweep_2026-07-02/sweep2/qwen3_5-2b/gsm8k/adapter_top10/adapter_config.json', '/content/tlm_job4/rishabh-tlm/cesplit_sweep_2026-07-02/sweep2/qwen3_5-2b/medicine/adapter_top10/adapter_config.json', '/content/tlm_job4/rishabh-tlm/cesplit_sweep_2026-07-02/sweep2/qwen3_5-2b/medicine/adapter_top20/adapter_config.json']
/content/tlm_job4/rishabh-tlm/cuda_ttl/ab_routing/README.md:loss, q+v r8 α16, lr 5e-5 cosine, 1 epoch; saves a **merged** model because vLLM 0.24
/content/tlm_job4/rishabh-tlm/docs/ARCHITECTURE.md:LoRA q+v, r8 α16, lr 5e-5 cosine, 1 epoch, seed 42 everywhere.


In [25]:
import json
sweep_cfg = json.load(open(f"{SWEEP['medicine'].parent}/medicine/adapter_top10/adapter_config.json"))
mine_cfg  = json.load(open(f"{ADAPT['medicine']}/adapter_config.json"))
for k in sorted(set(sweep_cfg)|set(mine_cfg)):
    if sweep_cfg.get(k)!=mine_cfg.get(k):
        print(f"{k}: sweep={sweep_cfg.get(k)}  mine={mine_cfg.get(k)}")

In [26]:
ADAPT = {d: SWEEP[d].parent/d/"adapter_top10" for d in DOMAINS}
# verify they load and aren't collapsed, before the full eval:
for d in DOMAINS:
    am,at=load_adapter(MODEL, ADAPT[d]); recs=BASE[d]["recs"]
    ap,_,_=hf_greedy(am,at,recs,list(range(6)),max_new=32); del am; torch.cuda.empty_cache()
    print(d, "adapted:", [ap[i].strip() for i in range(6)], "vs gold:", [goldof(recs[i]) for i in range(6)])

Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

gen:   0%|          | 0/1 [00:00<?, ?it/s]

medicine adapted: ['A', 'A', 'A', 'A', 'A', 'A'] vs gold: ['C', 'D', 'A', 'B', 'D', 'B']


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

gen:   0%|          | 0/1 [00:00<?, ?it/s]

logiqa adapted: ['A', 'A', 'A', 'A', 'A', 'A'] vs gold: ['D', 'D', 'C', 'A', 'B', 'A']


In [27]:
# does top-20% (bigger, less-extreme pool) avoid the constant-A collapse?
for d in ["medicine"]:
    a20 = SWEEP[d].parent/d/"adapter_top20"
    if a20.exists():
        am,at=load_adapter(MODEL,a20); recs=BASE[d]["recs"]
        ap,_,_=hf_greedy(am,at,recs,list(range(8)),max_new=32); del am; torch.cuda.empty_cache()
        print(d,"top20 adapted:",[ap[i].strip() for i in range(8)],"gold:",[goldof(recs[i]) for i in range(8)])

Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

gen:   0%|          | 0/1 [00:00<?, ?it/s]

medicine top20 adapted: ['A', 'A', 'A', 'A', 'A', 'A', 'A', 'A'] gold: ['C', 'D', 'A', 'B', 'D', 'B', 'A', 'C']


In [15]:
import subprocess, json
# the sweep's canonical eval path on its OWN shipped medicine adapter, 5 items
adir = SWEEP["medicine"].parent/"medicine"/"adapter_top10"
subprocess.run([sys.executable, str(REPO_DIR/"teams/rishabh/gen_mc_hf.py"),
    "--model", MODEL, "--adapter", str(adir),
    "--data", str(DATA["medicine"]), "--max-samples","5",
    "--output","/tmp/gmc_test.jsonl","--batch-size","5"], check=True)
for l in open("/tmp/gmc_test.jsonl"):
    r=json.loads(l); print("predict:", repr(r["predict"][:40]), "| gold:", r.get("label"))

predict: 'A\n' | gold: C
predict: 'A\n' | gold: D
predict: 'A\n' | gold: A
predict: 'A\n' | gold: B
predict: 'A\n' | gold: D


In [16]:
import json
# the sweep's OWN adapted preds for medicine top10 — what did IT record?
p = SWEEP["medicine"].parent/"medicine"/"preds_adapted_top10.jsonl"
print("exists:", p.exists())
if p.exists():
    for i,l in enumerate(open(p)):
        if i>=8: break
        r=json.loads(l); print("predict:", repr(str(r.get("predict"))[:40]), "| gold:", r.get("label"))
    # and its accuracy vs base
    import numpy as np, sys; sys.path.insert(0,str(SCRIPTS)); import grading
    preds=[json.loads(l) for l in open(p)]
    acc=np.mean([grading.is_correct(grading.extract(str(r["predict"]),"choice").value, str(r["label"]),"choice") for r in preds])
    print(f"\nsweep's own adapted top10 accuracy: {acc:.3f}")

exists: True
predict: 'A' | gold: C
predict: 'A' | gold: D
predict: 'A' | gold: A
predict: 'A' | gold: B
predict: 'A' | gold: D
predict: 'A' | gold: B
predict: 'A' | gold: A
predict: 'A' | gold: C

sweep's own adapted top10 accuracy: 0.256


In [17]:
import json, numpy as np, sys; sys.path.insert(0,str(SCRIPTS)); import grading
base=SWEEP["medicine"].parent/"medicine"
for f in ["preds_adapted_top10.jsonl","preds_adapted_top20.jsonl"]:
    p=base/f
    if p.exists():
        pr=[json.loads(l) for l in open(p)]
        letters=[grading.extract(str(r["predict"]),"choice").value for r in pr]
        acc=np.mean([grading.is_correct(l,str(r["label"]),"choice") for l,r in zip(letters,pr)])
        from collections import Counter
        print(f"{f}: acc={acc:.3f}  letter dist={Counter(letters).most_common(4)}")

preds_adapted_top10.jsonl: acc=0.256  letter dist=[('A', 5000)]
preds_adapted_top20.jsonl: acc=0.256  letter dist=[('A', 5000)]


In [28]:
for r in RESULT:
    e=r["eval"]; # recompute the conf-wrong slice size
    print(r["specialist"],"->",e, "base-wrong n=", ...) # how many base-wrong items in the top-half-within slice

medicine -> medicine base-wrong n= Ellipsis
medicine -> logiqa base-wrong n= Ellipsis
logiqa -> medicine base-wrong n= Ellipsis
logiqa -> logiqa base-wrong n= Ellipsis


In [29]:
import numpy as np, json
for spec in DOMAINS:
    for e in DOMAINS:
        b = BASE[e]
        recs = b["recs"]; idxs = list(range(len(recs)))
        within = np.array([b["within"].get(i, np.nan) for i in idxs])
        base_ok = np.array([b["base_ok"][i] for i in idxs])
        hi = within >= np.nanmedian(within)          # the "confident" half (top-half within-conf)
        n_slice   = int(hi.sum())                     # items in the confident slice
        n_basewrong = int((~base_ok & hi).sum())      # base-WRONG within the confident slice = the positive class
        print(f"{spec:9}->{e:9}  confident-slice={n_slice:4d}  base-wrong-in-slice={n_basewrong:4d}"
              f"  ({'OK' if n_basewrong>=50 else 'THIN — caveat this AUROC'})")

medicine ->medicine   confident-slice= 500  base-wrong-in-slice= 192  (OK)
medicine ->logiqa     confident-slice= 500  base-wrong-in-slice= 166  (OK)
logiqa   ->medicine   confident-slice= 500  base-wrong-in-slice= 192  (OK)
logiqa   ->logiqa     confident-slice= 500  base-wrong-in-slice= 166  (OK)
